# Energy network dispatch and reporting

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/equinor/neqsim/blob/master/examples/notebooks/energy_networks/01_energy_dispatch_and_reporting.ipynb)

This notebook demonstrates deterministic multi-source and multi-load allocation, priority-based shortage handling, balancing resources, cost, emissions, and auditable reports.

**Implementation dependencies:** Energy Networks v3 and PR #2613 for optional minimum-cost/minimum-emissions strategies.

```text
Wind ─┐
Grid ─┼─> Electrical EnergyBus ─> Critical load
GT   ─┘                         └> Flexible load
```

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

def find_neqsim_project_root():
    env_root = os.environ.get("NEQSIM_PROJECT_ROOT")
    candidates = [Path(env_root).resolve()] if env_root else []
    cwd = Path.cwd().resolve()
    candidates.extend([cwd] + list(cwd.parents))
    for candidate in candidates:
        if (candidate / "pom.xml").exists() and (candidate / "devtools" / "neqsim_dev_setup.py").exists():
            return candidate
    raise RuntimeError("Could not find NeqSim project root. Set NEQSIM_PROJECT_ROOT.")

PROJECT_ROOT = find_neqsim_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "devtools"))
from neqsim_dev_setup import neqsim_init, neqsim_classes

ns = neqsim_classes(neqsim_init(project_root=PROJECT_ROOT, recompile=False, verbose=True))
JClass = ns.JClass
print("NeqSim workspace classes loaded")

In [ ]:
EnergyBus = JClass("neqsim.process.equipment.stream.EnergyBus")
EnergyPort = JClass("neqsim.process.equipment.stream.EnergyPort")
EnergyType = JClass("neqsim.process.equipment.stream.EnergyType")
EnergyPortDirection = JClass("neqsim.process.equipment.stream.EnergyPortDirection")
EnergyPortMode = JClass("neqsim.process.equipment.stream.EnergyPortMode")
try:
    EnergyDispatchStrategy = JClass("neqsim.process.equipment.stream.EnergyDispatchStrategy")
except Exception:
    EnergyDispatchStrategy = None

## 2. Build a multi-source electrical network

In [ ]:
bus = EnergyBus("platform electrical bus", EnergyType.ELECTRICAL)

def port(owner, direction, mode):
    p = EnergyPort("power", EnergyType.ELECTRICAL, direction, mode)
    p.setOwnerName(owner)
    p.connect(bus)
    return p

wind = port("wind", EnergyPortDirection.OUTPUT, EnergyPortMode.CALCULATED)
gas_turbine = port("gas turbine", EnergyPortDirection.OUTPUT, EnergyPortMode.CALCULATED)
grid = port("grid import", EnergyPortDirection.OUTPUT, EnergyPortMode.SPECIFICATION)
critical = port("critical process load", EnergyPortDirection.INPUT, EnergyPortMode.SPECIFICATION)
flexible = port("flexible process load", EnergyPortDirection.INPUT, EnergyPortMode.SPECIFICATION)

wind.setDuty(8.0, "MW")
gas_turbine.setDuty(10.0, "MW")
grid.setRequestedPower(6.0, "MW")
critical.setRequestedPower(12.0, "MW")
flexible.setRequestedPower(8.0, "MW")

critical.setPriority(10)
flexible.setPriority(20)

wind.setEnergyPricePerMWh(0.0)
wind.setEmissionFactorKgPerMWh(0.0)
gas_turbine.setEnergyPricePerMWh(90.0)
gas_turbine.setEmissionFactorKgPerMWh(450.0)
grid.setEnergyPricePerMWh(120.0)
grid.setEmissionFactorKgPerMWh(250.0)

report = bus.solveBalance()
print(report.toJson())

## 3. Extract allocations

In [ ]:
import pandas as pd

rows = []
for item in report.getAllocations():
    rows.append({
        "participant": item.getParticipantName(),
        "requested_MW": item.getRequestedPower() / 1e6,
        "allocated_MW": item.getAllocatedPower() / 1e6,
        "unmet_MW": item.getUnmetPower() / 1e6,
        "curtailed_MW": item.getCurtailedPower() / 1e6,
    })
allocations = pd.DataFrame(rows)
allocations

## 4. Visualize dispatch

In [ ]:
import matplotlib.pyplot as plt

ax = allocations.set_index("participant")[["requested_MW", "allocated_MW"]].plot(kind="bar", figsize=(10, 4))
ax.set_ylabel("Power (MW)")
ax.set_title("Requested and allocated power")
ax.grid(axis="y")
plt.tight_layout()
plt.show()

**Interpretation.** Critical loads are served before flexible loads. The gap between request and allocation identifies curtailment or unmet demand at participant level.

In [ ]:
summary = pd.Series({
    "offered supply (MW)": report.getOfferedSupply()/1e6,
    "accepted supply (MW)": report.getAcceptedSupply()/1e6,
    "requested demand (MW)": report.getRequestedDemand()/1e6,
    "served demand (MW)": report.getServedDemand()/1e6,
    "unmet demand (MW)": report.getUnmetDemand()/1e6,
    "curtailed supply (MW)": report.getCurtailedSupply()/1e6,
})
ax = summary.plot(kind="bar", figsize=(9,4))
ax.set_ylabel("Power (MW)")
ax.set_title("Network balance")
ax.grid(axis="y")
plt.tight_layout()
plt.show()

**Interpretation.** Network-level KPIs distinguish shortage from curtailment. This is the primary operational view for capacity and reserve screening.

## 5. Compare dispatch strategies

In [ ]:
strategies = []
if EnergyDispatchStrategy is not None:
    for name in ["PRIORITY_PROPORTIONAL", "MINIMUM_COST", "MINIMUM_EMISSIONS"]:
        bus.setDispatchStrategy(getattr(EnergyDispatchStrategy, name))
        r = bus.solveBalance()
        strategies.append({
            "strategy": name,
            "cost_per_h": r.getOperatingCostPerHour(),
            "co2_kg_per_h": r.getCo2EmissionRate(),
            "served_MW": r.getServedDemand()/1e6,
        })
else:
    strategies.append({"strategy":"PRIORITY_PROPORTIONAL", "cost_per_h":report.getOperatingCostPerHour(),
                       "co2_kg_per_h":report.getCo2EmissionRate(), "served_MW":report.getServedDemand()/1e6})
strategy_df = pd.DataFrame(strategies)
strategy_df

In [ ]:
ax = strategy_df.set_index("strategy")[["cost_per_h", "co2_kg_per_h"]].plot(kind="bar", figsize=(10,4))
ax.set_ylabel("Rate per hour")
ax.set_title("Cost and emissions by dispatch strategy")
ax.grid(axis="y")
plt.tight_layout()
plt.show()

**Interpretation.** Minimum-cost and minimum-emissions dispatch can select different source combinations. The priority/proportional policy remains the transparent default.

## Summary

This notebook demonstrated:

- deterministic multi-party allocation;
- critical versus flexible demand priorities;
- shortage and curtailment reporting;
- cost and CO₂ accounting;
- optional merit-order dispatch.

For detailed electrical load flow, protection, and fault studies, couple NeqSim to a specialist power-system solver.